In [ ]:
import torch

# Check if GPU or MPS is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU is enabled: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS GPU is enabled.")
else:
    raise OSError(
        "No GPU or MPS device found. Please check your environment and ensure GPU or MPS support is configured."
    )

In [ ]:
from sentence_transformers import SentenceTransformer

# Load BGE embedding model from Hugging Face
embedding_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')

In [ ]:
#Load embedding model
def emb_text(text):
    embedding = embedding_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

In [ ]:
#Test snippet
test_embedding = emb_text("This is a test")
embedding_dim = len(test_embedding)
print(f"Embedding dimension: {embedding_dim}")
print(f"First 10 values: {test_embedding[:10]}")

In [ ]:
from docling_core.transforms.chunker import HierarchicalChunker
from docling.document_converter import DocumentConverter
from pymilvus import MilvusClient
from tqdm import tqdm
import json

converter = DocumentConverter()
chunker = HierarchicalChunker()

# Convert the input file to Docling Document
source = "docs/sample_docs/sample_statement.xlsx"
doc = converter.convert(source).document

# Perform hierarchical chunking
texts = [chunk.text for chunk in chunker.chunk(doc)]

In [ ]:
milvus_client = MilvusClient(uri="./milvus_pdf.db")
collection_name = "rag_pdf"

In [ ]:
data = []
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # Inner product distance
    consistency_level="Strong",  # Supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/consistency.md#Consistency-Level for more details.
)
for i, chunk in enumerate(tqdm(texts, desc="Processing chunks")):
    embedding = emb_text(chunk)
    data.append({"id": i, "vector": embedding, "text": chunk})

milvus_client.insert(collection_name=collection_name, data=data)

In [ ]:
#Query
question = (
    "Have the  The financial statements have been prepared in accordance with Indian Accounting Standards?"
)

In [ ]:
#Search for the question in the collection and retrieve the semantic top-6 matches.
search_res = milvus_client.search(
    collection_name=collection_name,
    data=[emb_text(question)],
    limit=6,
    search_params={"metric_type": "IP", "params": {}},
    output_fields=["text"],
)

In [ ]:
retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

In [ ]:
# Build context and generate response
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Configure GEMINI 2.5 Flash Lite
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure()
model = genai.GenerativeModel('gemini-2.5-flash-lite')

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

SYSTEM_PROMPT = """
Human: You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""
USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.

<context>
{context}
</context>

<question>
{question}
</question>
"""

response = model.generate_content([
    f"System: {SYSTEM_PROMPT}\n\nUser: {USER_PROMPT}"
])
print(response.text)